In [ ]:
"""
risk_labeling.py
================
Phase 2 - Step 1
Create financial risk labels from behavioral features.

Input:  data/processed/features.csv
Output: data/processed/features_labeled.csv
"""

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler


# ─────────────────────────────────────────────
# LOAD FEATURES
# ─────────────────────────────────────────────

def load_features(path="data/processed/features.csv"):
    df = pd.read_csv(path)
    print(f"✅ Loaded features: {len(df)} users")
    return df


# ─────────────────────────────────────────────
# CREATE RISK SCORE
# ─────────────────────────────────────────────

def create_risk_score(df):

    # Select core risk drivers
    core_features = [
        "expense_ratio_mean",
        "neg_savings_freq",
        "income_growth_rate"
    ]

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[core_features])

    df_scaled = pd.DataFrame(
        scaled,
        columns=["expense_scaled", "neg_savings_scaled", "growth_scaled"]
    )

    # Invert growth (declining income = higher risk)
    df_scaled["growth_scaled"] = 1 - df_scaled["growth_scaled"]

    # Weighted risk score
    df["risk_score"] = (
        0.4 * df_scaled["expense_scaled"] +
        0.4 * df_scaled["neg_savings_scaled"] +
        0.2 * df_scaled["growth_scaled"]
    )

    return df


# ─────────────────────────────────────────────
# CREATE RISK LABELS
# ─────────────────────────────────────────────

def assign_labels(df):

    # Use quantiles for balanced classes
    low_cut = df["risk_score"].quantile(0.3)
    high_cut = df["risk_score"].quantile(0.7)

    def label(score):
        if score >= high_cut:
            return "High"
        elif score <= low_cut:
            return "Low"
        else:
            return "Medium"

    df["risk_label"] = df["risk_score"].apply(label)

    print("\n📊 Risk Distribution:")
    print(df["risk_label"].value_counts())

    return df


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":

    print("\n🔄 Running Risk Labeling (Phase 2 - Step 1)...\n")

    os.makedirs("data/processed", exist_ok=True)

    df = load_features()
    df = create_risk_score(df)
    df = assign_labels(df)

    output_path = "data/processed/features_labeled.csv"
    df.to_csv(output_path, index=False)

    print(f"\n✅ Labeled dataset saved → {output_path}")